# 🔥 ClusterOps: Thermal GPU Balancer — GRPO Training

**Train an LLM (Llama-3.1-8B) to manage a GPU data center under adversarial thermal conditions using Group Relative Policy Optimization (GRPO).**

---

## 1. Install Dependencies

In [ ]:
# Install Unsloth and vLLM for fast training and inference
!pip install unsloth vllm -q
!pip install trl>=0.12.0 transformers>=4.45.0 -q
!pip install fastapi uvicorn requests pydantic openenv-core -q
!pip install matplotlib numpy -q
print('✅ Dependencies installed')

## 2. Clone & Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"

def setup_repo():
    if os.path.exists(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR])
    
    print(f"Cloning repository from {REPO_URL}...")
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Git clone failed! Error: {result.stderr}")
        return False
        
    os.chdir(REPO_DIR)
    print(f"✅ Successfully moved to {os.getcwd()}")
    return True

if setup_repo():
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    for i in range(15):
        time.sleep(2)
        try:
            resp = requests.get('http://localhost:8000/health', timeout=3)
            if resp.ok:
                print(f'✅ ClusterOps server online after {(i+1)*2}s')
                break
        except:
            pass
    else:
        print('❌ Server failed to start.')

## 3. Baseline Agent

In [ ]:
ENV_URL = 'http://localhost:8000'
def env_reset(scenario='01_baseline'): return requests.post(f'{ENV_URL}/reset', json={'scenario': scenario}).json()
def env_step(action): return requests.post(f'{ENV_URL}/step', json=action).json()
def env_rubric(): return requests.get(f'{ENV_URL}/grader/rubric').json()

print('Running baseline test...')
r, rub = env_reset(), env_rubric()
print(f'📊 Initial Score: {rub["total"]:.3f}')

## 4. Load LLM with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False, # Set to True ONLY if vLLM is successfully installed
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('✅ Model loaded with LoRA adapters')

## 5. GRPO Reward Functions

In [ ]:
import json
SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Output ONE JSON action only.
  {\"action_type\": \"allocate\", \"job_id\": \"job_1\", \"node_id\": 2}
  {\"action_type\": \"wait\"}"""

def parse_action(text):
    try:
        start, end = text.index('{'), text.rindex('}') + 1
        return json.loads(text[start:end])
    except: return {'action_type': 'wait'}

@torch.no_grad()
def run_llm_episode(model, tokenizer, scenario='01_baseline'):
    data = env_reset(scenario)
    total_reward = 0.0
    while not data.get('done', False):
        obs, meta = data.get('observation', {}), data.get('metadata', {})
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\nStep {meta.get('step')}: {obs}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, temperature=0.5)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        data = env_step(parse_action(text))
        total_reward += data.get('reward', 0)
    return total_reward, env_rubric()

print('✅ GRPO logic ready')

## 6. Training & Results

In [ ]:
print("Starting training episodes...")
for ep in range(10):
    r, rub = run_llm_episode(model, tokenizer, '01_baseline')
    print(f"  Episode {ep+1}: Reward={r:.1f} Score={rub['total']:.3f}")